# Mini-Projeto ETL — IQVIA Data
## Fase 2: Transformação — Camada Silver

**Objetivo:** Ler os arquivos brutos da camada Bronze, aplicar transformações de qualidade e salvar os dados limpos e padronizados na camada **Silver** do GCS.

| Transformação | Descrição |
|---------------|-----------|
| Renomeação de colunas | Padronização para snake_case sem caracteres especiais |
| Tipos de dados | Garantia dos tipos corretos (int, float, str) |
| Valores nulos | Preenchimento com 0 para colunas numéricas de volume |
| Duplicatas | Remoção de linhas completamente duplicadas |
| Extração de IDs | Separação do código numérico do campo BRICK |
| Enriquecimento | Join entre vendas e mapeamento de filiais |
| Coluna de auditoria | Adição de `dt_processamento` com timestamp da carga Silver |

**Bucket GCS:** `etl-iqvia-data-lake-augusto`  
**Origem:** `bronze/iqvia/<YYYY>/<MM>/<DD>/`  
**Destino:** `silver/iqvia/<YYYY>/<MM>/<DD>/`

## 1. Instalação e Importação de Dependências

In [10]:
%pip install -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for pyarrow (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [877 lines of output]
      C:\Users\onlyg\AppData\Local\Temp\pip-build-env-djtz7xqn\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsDeprecationWarning: `project.license` as a TOML table is deprecated
      !!
      
              ********************************************************************************
              Please use a simple string containing a SPDX expression for `project.license`. You can also use `project.license-files`. (Both options available on setuptools>=77.0.0).
      
              By 2027-Feb-18, you need to update your project and remove deprecated calls
              or your builds will no longer be supported.
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ************************************************

In [11]:
import io
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
from google.cloud import storage

## 2. Configurações do Pipeline

In [12]:
GCS_PROJECT_ID     = "proc-de-dados-iqvia-com-scd"
GCS_BUCKET_NAME    = "etl-iqvia-data-lake-augusto"
GCS_BRONZE_PREFIX  = "bronze/iqvia"
GCS_SILVER_PREFIX  = "silver/iqvia"

LOAD_TIMESTAMP      = datetime.now(timezone.utc)
LOAD_DATE_DISPLAY   = LOAD_TIMESTAMP.strftime("%d/%m/%Y")

# Partição da carga Bronze a ser processada.
# Por padrão usa a data de hoje. Para processar uma carga anterior,
# defina manualmente no formato "YYYY/MM/DD", ex: "2026/03/18"
BRONZE_DATE_PARTITION = "2026/03/18"  # None = usa hoje

if BRONZE_DATE_PARTITION is None:
    BRONZE_DATE_PARTITION = LOAD_TIMESTAMP.strftime("%Y/%m/%d")

# Partição de destino Silver (sempre usa hoje como data de processamento)
SILVER_DATE_PARTITION = LOAD_TIMESTAMP.strftime("%Y/%m/%d")

# Padrões para identificar cada arquivo — usa prefixo do nome, não nome exato
# Arquivo de vendas: começa com "MS_" (ex: MS_12_2022, MS_01_2023...)
# Arquivo de filiais: contém "filial" no nome
PATTERN_VENDAS  = "MS_"
PATTERN_FILIAIS = "filial"

print(f"Projeto GCP         : {GCS_PROJECT_ID}")
print(f"Bucket              : {GCS_BUCKET_NAME}")
print(f"Origem (Bronze)     : {GCS_BRONZE_PREFIX}/{BRONZE_DATE_PARTITION}/")
print(f"Destino (Silver)    : {GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/")
print(f"Data processamento  : {LOAD_DATE_DISPLAY}")
print(f"Load timestamp      : {LOAD_TIMESTAMP.strftime('%d/%m/%Y %H:%M:%S UTC')}")

Projeto GCP         : proc-de-dados-iqvia-com-scd
Bucket              : etl-iqvia-data-lake-augusto
Origem (Bronze)     : bronze/iqvia/2026/03/18/
Destino (Silver)    : silver/iqvia/2026/03/20/
Data processamento  : 20/03/2026
Load timestamp      : 20/03/2026 12:26:55 UTC


## 3. Conexão com o GCS

In [13]:
# Autenticação via Application Default Credentials (ADC).
# Execute previamente no terminal (requer Google Cloud SDK instalado):
#   gcloud auth application-default login
#   gcloud auth application-default set-quota-project proc-de-dados-iqvia-com-scd

gcs_client = storage.Client(project=GCS_PROJECT_ID)
bucket     = gcs_client.bucket(GCS_BUCKET_NAME)

print(f"Conectado ao GCS | Projeto: {gcs_client.project}")
print(f"Bucket: gs://{GCS_BUCKET_NAME}")

Conectado ao GCS | Projeto: proc-de-dados-iqvia-com-scd
Bucket: gs://etl-iqvia-data-lake-augusto


## 4. Extração — Leitura dos Arquivos da Camada Bronze

In [14]:
def discover_bronze_files(
    gcs_client: storage.Client,
    bucket_name: str,
    prefix: str
) -> dict[str, str]:
    """Lista os blobs da partição Bronze e retorna um dicionário nome → path completo."""
    blobs = gcs_client.list_blobs(bucket_name, prefix=prefix)
    return {
        blob.name.split("/")[-1]: blob.name
        for blob in blobs
        if not blob.name.endswith("_manifest.json")
    }


def find_file_by_pattern(files: dict[str, str], pattern: str) -> str:
    """Localiza o path de um arquivo cujo nome contenha o padrão informado.

    Raises:
        FileNotFoundError: se nenhum arquivo corresponder ao padrão.
        ValueError: se mais de um arquivo corresponder ao padrão.
    """
    matches = {name: path for name, path in files.items() if pattern.lower() in name.lower()}
    if not matches:
        raise FileNotFoundError(
            f"Nenhum arquivo com padrao '{pattern}' encontrado.\n"
            f"Arquivos disponiveis: {list(files.keys())}\n"
            f"Verifique se BRONZE_DATE_PARTITION aponta para a particao correta."
        )
    if len(matches) > 1:
        raise ValueError(f"Multiplos arquivos correspondem ao padrao '{pattern}': {list(matches.keys())}")
    return next(iter(matches.values()))


def read_excel_from_gcs(bucket: storage.Bucket, blob_path: str) -> pd.DataFrame:
    """Lê um arquivo .xlsx diretamente do GCS e retorna um DataFrame."""
    blob = bucket.blob(blob_path)
    data = blob.download_as_bytes()
    return pd.read_excel(io.BytesIO(data))


# Descoberta dinâmica dos arquivos na partição Bronze
bronze_prefix = f"{GCS_BRONZE_PREFIX}/{BRONZE_DATE_PARTITION}/"
bronze_files  = discover_bronze_files(gcs_client, GCS_BUCKET_NAME, bronze_prefix)

if not bronze_files:
    raise FileNotFoundError(
        f"Nenhum arquivo encontrado em gs://{GCS_BUCKET_NAME}/{bronze_prefix}\n"
        f"Ajuste BRONZE_DATE_PARTITION na célula de configurações para a data correta."
    )

print(f"Arquivos encontrados em gs://{GCS_BUCKET_NAME}/{bronze_prefix}")
for name in bronze_files:
    print(f"  {name}")

# Localiza cada arquivo pelo padrão configurado
bronze_vendas_path  = find_file_by_pattern(bronze_files, PATTERN_VENDAS)
bronze_filiais_path = find_file_by_pattern(bronze_files, PATTERN_FILIAIS)

print(f"\nArquivo de vendas  : {bronze_vendas_path}")
print(f"Arquivo de filiais : {bronze_filiais_path}")

# Leitura dos DataFrames
df_vendas  = read_excel_from_gcs(bucket, bronze_vendas_path)
df_filiais = read_excel_from_gcs(bucket, bronze_filiais_path)

print(f"\ndf_vendas  — shape: {df_vendas.shape}")
print(f"df_filiais — shape: {df_filiais.shape}")

Arquivos encontrados em gs://etl-iqvia-data-lake-augusto/bronze/iqvia/2026/03/18/
  MS_12_2022_sample.xlsx
  filial-brick_sample.xlsx

Arquivo de vendas  : bronze/iqvia/2026/03/18/MS_12_2022_sample.xlsx
Arquivo de filiais : bronze/iqvia/2026/03/18/filial-brick_sample.xlsx

df_vendas  — shape: (9, 6)
df_filiais — shape: (586, 2)

df_vendas  — shape: (9, 6)
df_filiais — shape: (586, 2)


## 5. Transformações — Camada Silver

### 5.1 Tabela de Filiais (`filial-brick`)

In [15]:
def transform_filiais(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa e padroniza o DataFrame de mapeamento brick → filial."""
    df = df.copy()

    # Renomeia colunas para snake_case
    df.columns = ["brick", "cod_filial"]

    # Remove duplicatas
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicatas removidas (filiais): {before - len(df)}")

    # Remove espaços extras nos valores de texto
    df["brick"] = df["brick"].str.strip()

    # Extrai o código numérico do brick (ex: '1147 - CAMPO GRANDE' → 1147)
    df["cod_brick"] = df["brick"].str.extract(r'^(\d+)').astype("Int64")

    # Extrai o nome descritivo do brick
    df["desc_brick"] = df["brick"].str.extract(r'^\d+\s*-\s*(.+)$').iloc[:, 0].str.strip()

    # Garante tipo correto de cod_filial
    df["cod_filial"] = df["cod_filial"].astype("Int64")

    # Timestamp de processamento
    df["dt_processamento"] = LOAD_TIMESTAMP.strftime("%d/%m/%Y %H:%M:%S UTC")

    return df[["cod_brick", "desc_brick", "brick", "cod_filial", "dt_processamento"]]


df_filiais_silver = transform_filiais(df_filiais)

print(f"Shape final: {df_filiais_silver.shape}")
print(f"Tipos:\n{df_filiais_silver.dtypes}")
display(df_filiais_silver.head())

Duplicatas removidas (filiais): 0
Shape final: (586, 5)
Tipos:
cod_brick           Int64
desc_brick            str
brick                 str
cod_filial          Int64
dt_processamento      str
dtype: object


,cod_brick,desc_brick,brick,cod_filial,dt_processamento
0,1,SE,1 - SE,402,20/03/2026 12:26:55 UTC
1,92,CHAC.SANTO ANTONIO,92 - CHAC.SANTO ANTONIO,454,20/03/2026 12:26:55 UTC
2,181,REGISTRO,181 - REGISTRO,423,20/03/2026 12:26:55 UTC
3,257,S.J.RIO PRETO-CENTRO,257 - S.J.RIO PRETO-CENTRO,394,20/03/2026 12:26:55 UTC
4,271,S.J.RIO PRETO-V.S.MANOEL,271 - S.J.RIO PRETO-V.S.MANOEL,417,20/03/2026 12:26:55 UTC


### 5.2 Tabela de Vendas (`MS_12_2022`)

In [16]:
def transform_vendas(df: pd.DataFrame) -> pd.DataFrame:
    """Limpa e padroniza o DataFrame de vendas IQVIA."""
    df = df.copy()

    # Renomeia colunas para snake_case legível
    df.columns = [
        "brick",
        "ean",
        "cod_prod_catarinense",
        "vol_si_concorrente",
        "vol_so_concorrente",
        "vol_so_preco_popular",
    ]

    # Remove duplicatas
    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicatas removidas (vendas): {before - len(df)}")

    # Remove espaços extras em texto
    df["brick"] = df["brick"].str.strip()

    # Extrai código numérico do brick para join posterior
    df["cod_brick"] = df["brick"].str.extract(r'^(\d+)').astype("Int64")

    # Padroniza tipos numéricos
    df["ean"]                 = df["ean"].astype("Int64")
    df["cod_prod_catarinense"] = df["cod_prod_catarinense"].astype("Int64")

    # Preenche nulos em colunas de volume com 0 (ausência de venda = zero unidades)
    volume_cols = ["vol_si_concorrente", "vol_so_concorrente", "vol_so_preco_popular"]
    nulos_antes = df[volume_cols].isnull().sum()
    df[volume_cols] = df[volume_cols].fillna(0)
    print(f"Nulos preenchidos com 0:\n{nulos_antes}")

    # Converte colunas de volume para inteiro (unidades não são fracionadas)
    for col in volume_cols:
        df[col] = df[col].astype("Int64")

    # Adiciona referência de competência (mês/ano do arquivo: Dezembro/2022)
    df["competencia"] = "12/2022"

    # Timestamp de processamento
    df["dt_processamento"] = LOAD_TIMESTAMP.strftime("%d/%m/%Y %H:%M:%S UTC")

    return df[[
        "cod_brick", "brick", "ean", "cod_prod_catarinense",
        "vol_si_concorrente", "vol_so_concorrente", "vol_so_preco_popular",
        "competencia", "dt_processamento"
    ]]


df_vendas_silver = transform_vendas(df_vendas)

print(f"\nShape final: {df_vendas_silver.shape}")
print(f"Tipos:\n{df_vendas_silver.dtypes}")
display(df_vendas_silver.head())

Duplicatas removidas (vendas): 0
Nulos preenchidos com 0:
vol_si_concorrente      5
vol_so_concorrente      0
vol_so_preco_popular    2
dtype: int64

Shape final: (9, 9)
Tipos:
cod_brick               Int64
brick                     str
ean                     Int64
cod_prod_catarinense    Int64
vol_si_concorrente      Int64
vol_so_concorrente      Int64
vol_so_preco_popular    Int64
competencia               str
dt_processamento          str
dtype: object


,cod_brick,brick,ean,cod_prod_catarinense,vol_si_concorrente,vol_so_concorrente,vol_so_preco_popular,competencia,dt_processamento
0,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC
1,1147,1147 - CAMPO GRANDE - CENTRO,42110200,630693,0,7,5,12/2022,20/03/2026 12:26:55 UTC
2,1147,1147 - CAMPO GRANDE - CENTRO,42176763,687607,0,4,0,12/2022,20/03/2026 12:26:55 UTC
3,1147,1147 - CAMPO GRANDE - CENTRO,42277217,739189,7,85,23,12/2022,20/03/2026 12:26:55 UTC
4,1147,1147 - CAMPO GRANDE - CENTRO,42355014,735367,0,2,1,12/2022,20/03/2026 12:26:55 UTC


### 5.3 Enriquecimento — Join Vendas × Filiais

In [17]:
df_silver = df_vendas_silver.merge(
    df_filiais_silver[["cod_brick", "cod_filial", "desc_brick"]],
    on="cod_brick",
    how="left"
)

sem_filial = df_silver["cod_filial"].isnull().sum()
print(f"Registros sem filial mapeada: {sem_filial}")
print(f"Shape após join: {df_silver.shape}")
display(df_silver.head())

Registros sem filial mapeada: 0
Shape após join: (45, 11)


,cod_brick,brick,ean,cod_prod_catarinense,vol_si_concorrente,vol_so_concorrente,vol_so_preco_popular,competencia,dt_processamento,cod_filial,desc_brick
0,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,950,CAMPO GRANDE - CENTRO
1,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,373,CAMPO GRANDE - CENTRO
2,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,386,CAMPO GRANDE - CENTRO
3,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,478,CAMPO GRANDE - CENTRO
4,1147,1147 - CAMPO GRANDE - CENTRO,32689150,741013,0,0,5,12/2022,20/03/2026 12:26:55 UTC,672,CAMPO GRANDE - CENTRO


## 6. Validação da Qualidade dos Dados

In [18]:
def validate_dataframe(df: pd.DataFrame, name: str) -> dict:
    """Executa checagens de qualidade e retorna um resumo."""
    total_rows     = len(df)
    total_dupes    = df.duplicated().sum()
    nulls_por_col  = df.isnull().sum().to_dict()
    total_nulls    = sum(nulls_por_col.values())

    print(f"\n{'='*50}")
    print(f" Validação: {name}")
    print(f"{'='*50}")
    print(f"  Linhas       : {total_rows}")
    print(f"  Duplicatas   : {total_dupes}")
    print(f"  Nulos totais : {total_nulls}")
    if total_nulls > 0:
        print(f"  Nulos por coluna: {nulls_por_col}")
    print(f"  Status       : {'OK' if total_dupes == 0 and total_nulls == 0 else 'ATENCAO — verificar'}")

    return {"tabela": name, "linhas": total_rows, "duplicatas": int(total_dupes), "nulos": total_nulls}


validation_results = [
    validate_dataframe(df_vendas_silver,  "vendas_silver"),
    validate_dataframe(df_filiais_silver, "filiais_silver"),
    validate_dataframe(df_silver,         "silver_enriquecido"),
]


 Validação: vendas_silver
  Linhas       : 9
  Duplicatas   : 0
  Nulos totais : 0
  Status       : OK

 Validação: filiais_silver
  Linhas       : 586
  Duplicatas   : 0
  Nulos totais : 0
  Status       : OK

 Validação: silver_enriquecido
  Linhas       : 45
  Duplicatas   : 0
  Nulos totais : 0
  Status       : OK


## 7. Carga — Salvando Parquet na Camada Silver do GCS

In [19]:
def save_parquet_to_gcs(
    bucket: storage.Bucket,
    df: pd.DataFrame,
    gcs_path: str,
    load_timestamp: datetime
) -> dict:
    """Serializa o DataFrame como Parquet e faz upload para o GCS.

    Parquet é o formato padrão para camadas Silver/Gold:
    compressão eficiente, suporte a tipos e leitura colunar.
    """
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    buffer.seek(0)

    blob = bucket.blob(gcs_path)
    blob.metadata = {
        "layer": "silver",
        "pipeline": "iqvia-etl",
        "load_timestamp": load_timestamp.isoformat(),
        "rows": str(len(df)),
        "columns": str(len(df.columns)),
    }
    blob.upload_from_file(buffer, content_type="application/octet-stream")

    size_kb = round(buffer.getbuffer().nbytes / 1024, 2)
    print(f"Salvo: gs://{bucket.name}/{gcs_path}  ({size_kb} KB | {len(df)} linhas)")

    return {
        "gcs_path": f"gs://{bucket.name}/{gcs_path}",
        "rows": len(df),
        "size_kb": size_kb,
        "load_timestamp": load_timestamp.isoformat(),
        "status": "success",
    }


silver_manifest = []

silver_manifest.append(save_parquet_to_gcs(
    bucket, df_vendas_silver,
    f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/vendas.parquet",
    LOAD_TIMESTAMP
))

silver_manifest.append(save_parquet_to_gcs(
    bucket, df_filiais_silver,
    f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/filiais.parquet",
    LOAD_TIMESTAMP
))

silver_manifest.append(save_parquet_to_gcs(
    bucket, df_silver,
    f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/vendas_enriquecido.parquet",
    LOAD_TIMESTAMP
))

Salvo: gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/vendas.parquet  (6.21 KB | 9 linhas)
Salvo: gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/filiais.parquet  (12.24 KB | 586 linhas)
Salvo: gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/vendas_enriquecido.parquet  (7.48 KB | 45 linhas)
Salvo: gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/vendas_enriquecido.parquet  (7.48 KB | 45 linhas)


## 8. Relatório Final da Transformação

In [20]:
df_report = pd.DataFrame(silver_manifest)

print("=" * 60)
print("RELATORIO DE TRANSFORMACAO — CAMADA SILVER")
print("=" * 60)
print(f"  Pipeline  : IQVIA ETL")
print(f"  Bucket    : gs://{GCS_BUCKET_NAME}")
print(f"  Origem    : {GCS_BRONZE_PREFIX}/{BRONZE_DATE_PARTITION}/")
print(f"  Destino   : {GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/")
print(f"  Timestamp : {LOAD_TIMESTAMP.strftime('%d/%m/%Y %H:%M:%S UTC')}")
print(f"  Arquivos  : {len(silver_manifest)}")
print(f"  Sucesso   : {df_report[df_report['status'] == 'success'].shape[0]}")
print("=" * 60)
display(df_report[["gcs_path", "rows", "size_kb", "status"]])

RELATORIO DE TRANSFORMACAO — CAMADA SILVER
  Pipeline  : IQVIA ETL
  Bucket    : gs://etl-iqvia-data-lake-augusto
  Origem    : bronze/iqvia/2026/03/18/
  Destino   : silver/iqvia/2026/03/20/
  Timestamp : 20/03/2026 12:26:55 UTC
  Arquivos  : 3
  Sucesso   : 3


,gcs_path,rows,size_kb,status
0,gs://etl-iqvia-data-lake-augusto/silver/iqvia/...,9,6.21,success
1,gs://etl-iqvia-data-lake-augusto/silver/iqvia/...,586,12.24,success
2,gs://etl-iqvia-data-lake-augusto/silver/iqvia/...,45,7.48,success


## 9. Verificação dos Arquivos no Bucket

In [21]:
prefix_filter = f"{GCS_SILVER_PREFIX}/{SILVER_DATE_PARTITION}/"
blobs = list(gcs_client.list_blobs(GCS_BUCKET_NAME, prefix=prefix_filter))

print(f"Conteudo de gs://{GCS_BUCKET_NAME}/{prefix_filter}")
print("-" * 60)

if not blobs:
    print("  (nenhum arquivo encontrado)")
else:
    for blob in blobs:
        size_kb = round(blob.size / 1024, 2)
        updated = blob.updated.strftime("%d/%m/%Y %H:%M:%S UTC")
        print(f"  {blob.name.split('/')[-1]}  |  {size_kb} KB  |  {updated}")

print("\nTransformacao Silver finalizada. Pipeline pronto para a proxima fase (Gold / SCD).")

Conteudo de gs://etl-iqvia-data-lake-augusto/silver/iqvia/2026/03/20/
------------------------------------------------------------
  filiais.parquet  |  12.24 KB  |  20/03/2026 12:27:06 UTC
  vendas.parquet  |  6.21 KB  |  20/03/2026 12:27:05 UTC
  vendas_enriquecido.parquet  |  7.48 KB  |  20/03/2026 12:27:06 UTC

Transformacao Silver finalizada. Pipeline pronto para a proxima fase (Gold / SCD).
